# FH-04 · Indian adaptation, robustness, performance, regression baseline (Phases 8, 9, 12.1, 12.4)
Needs the GPU and the FH-01 results. Parts: **A** language slices (no GPU) · **B** robustness · **C** speed/VRAM/quantization · **D** general-capability baseline for forgetting tests.

Stage-2 (Indian adapter) is **training** work. This notebook builds the ICAD-style test and the regression suite so that, when an adapter exists, you only change `TAG`/`ADAPTER` and rerun.

In [ ]:
# ---- Colab setup: upload futurehuman_testing_suite.zip when asked (once per session) ----
import os, sys, subprocess
SUITE = "/content/futurehuman_testing_suite"
if not os.path.isdir(SUITE):
    from google.colab import files
    up = files.upload()
    subprocess.run(["unzip", "-o", "-q", list(up)[0], "-d", "/content"], check=True)
os.chdir(SUITE); sys.path.insert(0, SUITE)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "accelerate", "bitsandbytes", "tabulate"])
print("ready in", os.getcwd())

In [ ]:
TAG = "base"          # change to "stage1" / "stage2" when you test a trained adapter (same notebook, same tests)
ADAPTER = None        # e.g. "/content/drive/MyDrive/adapters/stage1"  (None = untouched base model)
QUANT = "4bit"        # 4bit | 8bit | fp16
from fh import env
from fh.model import load_qwen, make_gen
env.set_seed(0)
model, tok = load_qwen("Qwen/Qwen3-4B", quant=QUANT, adapter=ADAPTER)
gen = make_gen(model, tok)
info = env.save_snapshot(f"results/env_{TAG}.json", model=model, model_id="Qwen/Qwen3-4B", quant=QUANT, seed=0,
                         extra=dict(adapter=ADAPTER))
info

In [ ]:
import json, numpy as np, pandas as pd
pd.set_option("display.max_colwidth", 120)
from fh.dataset import load_actionability
from fh.evaluate import run_cases, save_run
from fh.prompts import select_fewshot
from fh.metrics import safety, fmt, slice_report
from fh.report import update_capabilities, load_runs
DATA = "data/actionability_v1_adjudicated.jsonl" if os.path.exists("data/actionability_v1_adjudicated.jsonl") else "data/actionability_v1.jsonl"
cases = load_actionability(DATA); dev = [c for c in cases if c["split"] == "dev"]; test = [c for c in cases if c["split"] == "test"]
fs = select_fewshot(dev, n=(3, 3, 2), seed=0)
clean = load_runs("results", split="test")[f"{TAG}_B_fewshot_test"]

## A · Phase 8 - English vs Hinglish vs Hindi (same model, same metrics)

In [ ]:
sl = slice_report(clean, "lang"); print(sl.round(3).to_string(index=False))
en = sl[sl.lang == "en"].accuracy.iloc[0]; ind = clean[clean.lang != "en"]; ind_acc = (ind.gold_label == ind.pred_label).mean()
gap = (en - ind_acc) * 100; print(f"\naccuracy gap English vs Indian-language slices: {gap:.1f} points")
p95 = float(clean.latency_s.quantile(.95))
print("\nIndian minimal-pair groups (P5 Hinglish, P6 Hindi, P9 Hinglish):")
clean[clean.pair.isin(["P5", "P6", "P9"])][["pair", "id", "gold_label", "pred_label"]].sort_values(["pair", "id"])

In [ ]:
tr = pd.read_csv(f"results/timeres_{TAG}.csv"); print("time resolution exact-match:", tr.ok.mean().round(3))
tr[tr.text.str.contains("Kal|Parso|Next", case=False)][["text", "gold", "pred", "ok"]]

## B · Phase 12.1 - robustness (labels must not change when the surface form changes)

In [ ]:
from fh.robustness import perturb_cases, speaker_swap_last, PERTURBATIONS
rows, perturbed = [], {}
base_s = safety(clean.gold_label.tolist(), clean.pred_label.tolist())
rows.append(dict(condition="clean", accuracy=base_s["accuracy"]["rate"], false_action=base_s["false_action"]["rate"], missed=base_s["missed_action"]["rate"]))
for name in PERTURBATIONS:
    d = run_cases(perturb_cases(test, name), gen, "A1_improved", fewshot=fs, progress=False); perturbed[name] = d
    s_ = safety(d.gold_label.tolist(), d.pred_label.tolist())
    rows.append(dict(condition=name, accuracy=s_["accuracy"]["rate"], false_action=s_["false_action"]["rate"], missed=s_["missed_action"]["rate"]))
rb = pd.DataFrame(rows); rb["acc_drop_pts"] = ((rb.accuracy.iloc[0] - rb.accuracy) * 100).round(1); rb.round(3)

In [ ]:
inj = perturbed["prompt_injection"]; neg = inj[inj.gold_label != "actionable"]
pushed = int((neg.pred_label == "actionable").sum())
print(f"PROMPT INJECTION: {pushed}/{len(neg)} non-actionable/ambiguous cases pushed to 'actionable'")
swapped = [speaker_swap_last(c) for c in test]
sw = run_cases(swapped, gen, "A1_improved", fewshot=fs, progress=False)
merged = sw.merge(clean[["id", "pred_label"]], on="id", suffixes=("_swapped", "_clean"))
print("SPEAKER SWAP (same words, other speaker): decision changed in", round((merged.pred_label_swapped != merged.pred_label_clean).mean() * 100, 1), "% of cases")

## C · Phase 12.4 - speed, VRAM, quantization (Colab T4 is only a proxy for the target laptop)

In [ ]:
import torch, gc
from fh.perf import time_generations, pad_turns
from fh.prompts import build_messages
from fh.model import load_qwen, make_gen
del model, gen; gc.collect(); torch.cuda.empty_cache()
perf = []
for q in ["4bit", "8bit", "fp16"]:
    m_, t_ = load_qwen("Qwen/Qwen3-4B", quant=q); g_ = make_gen(m_, t_)
    d = run_cases(dev[:30], g_, "A1_improved", fewshot=None, progress=False)
    acc = (d.gold_label == d.pred_label).mean()
    tm = time_generations(g_, [build_messages(c, "A1_improved") for c in dev[:20]])
    perf.append(dict(quant=q, dev30_accuracy=round(acc, 3), schema_valid=round(d.schema_valid.mean(), 3), **{k: round(v, 2) for k, v in tm.items() if k != "n"}))
    del m_, g_, t_; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(perf)

In [ ]:
model, tok = load_qwen("Qwen/Qwen3-4B", quant=QUANT, adapter=ADAPTER); gen = make_gen(model, tok)
ctx = []
for L in [128, 512, 1024, 2048]:
    msgs = [build_messages(pad_turns(c, L, tok), "A1_improved") for c in dev[:5]]
    tm = time_generations(gen, msgs); ctx.append(dict(target_prompt_tokens=L, **{k: round(v, 2) for k, v in tm.items() if k != "n"}))
ctx = pd.DataFrame(ctx); print(ctx.to_string(index=False))
p95 = float(clean.latency_s.quantile(.95)); print("\nmodel-only p95 latency on test (s):", round(p95, 2), "| the pipeline also needs VAD + ASR + gate inside the 5 s capture interval")

## D · Phase 9 - regression suite baseline
28 auto-checked general prompts (math, facts, instruction following, chat, Hindi/Hinglish). The `chat` group fails if the model starts answering casual messages with JSON -
a typical symptom of over-specialisation. After training: set `TAG="stage1"`, `ADAPTER=...`, rerun FH-01/02/04, then compare below.

In [ ]:
from fh.regression import run_general, general_summary, compare_runs
g = run_general(gen); gs = general_summary(g)
print("general capability:", fmt(gs["overall"]), "| by category:", {k: round(v, 2) for k, v in gs["by_category"].items()})
pd.DataFrame(g)[["category", "ok", "prompt", "output"]].query("not ok")

In [ ]:
rb_caps = dict(general_acc=dict(value=gs["overall"]["rate"], k=gs["overall"]["k"], n=gs["overall"]["n"]),
               indian_gap_points=dict(value=float(gap)), latency_p95_s=dict(value=p95),
               robustness_worst_drop_pts=dict(value=float(rb.acc_drop_pts.max())))
caps = update_capabilities(TAG, **rb_caps); caps

In [ ]:
# ---- AFTER TRAINING: compare any two tags (e.g. base vs stage1). Needs the same keys in both capability files.
from fh.report import load_capabilities
BEFORE, AFTER = "base", "stage1"
b, a = load_capabilities(BEFORE), load_capabilities(AFTER)
if a:
    keys = ["actionability_macro_f1", "false_action_rate", "schema_valid_rate", "intent_accuracy", "general_acc", "multiwoz_jga", "massive_intent_acc_en", "massive_intent_acc_hi"]
    table, verdict = compare_runs({k: b[k] for k in keys if k in b}, {k: a[k] for k in keys if k in a}, tolerance_points=2.0)
    print(table.to_string(index=False)); print("\nCHECKPOINT VERDICT:", verdict)
else:
    print("no", AFTER, "capabilities yet - run the notebooks with TAG='%s' first" % AFTER)